# Quick-Start 03: Deep Q-Network (DQN) on CartPole-v1

**Estimated time: under 2 minutes**

## Learning Objectives

By the end of this notebook you will:
1. Understand why experience replay and a target network are necessary for stable DQN training
2. Implement a complete DQN from scratch — replay buffer, Q-network, target network, training loop
3. Observe how DQN combines value-based Q-learning with neural function approximation

## Prerequisites

- Quick-Start 01 (tabular Q-learning, Bellman update)
- Quick-Start 02 (PyTorch basics: linear layers, optimisers, backprop)

## Dependencies

```
pip install gymnasium numpy matplotlib torch
```

---

## Setup

We seed everything and configure key hyperparameters as named constants so they are
easy to find and modify.

In [ ]:
# Purpose: Imports, seeds, and hyperparameters
# Key Concept: Centralise hyperparameters — changing one value should not require hunting through the notebook

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from collections import deque
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# --- Hyperparameters ---
N_EPISODES      = 300      # training episodes (< 1 min on CPU)
BUFFER_SIZE     = 10_000   # replay buffer capacity
BATCH_SIZE      = 64       # transitions sampled per update
GAMMA           = 0.99     # discount factor
LR              = 1e-3     # Adam learning rate
EPSILON_START   = 1.0      # initial exploration probability
EPSILON_END     = 0.05     # minimum exploration probability
EPSILON_DECAY   = 0.995    # multiplicative decay per episode
TARGET_UPDATE   = 10       # copy online -> target network every N episodes
MIN_BUFFER_SIZE = 500      # do not start training until buffer has this many transitions

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device:  {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Gymnasium: {gym.__version__}")

## Why DQN? The Problems with Naive Deep Q-Learning

Replacing the Q-table with a neural network sounds straightforward, but two instabilities
arise immediately:

1. **Correlated samples:** consecutive environment steps are highly correlated.
   Training on correlated mini-batches destabilises stochastic gradient descent.
   
   **Fix: Experience Replay** — store transitions in a buffer and sample random mini-batches.

2. **Moving targets:** we update $Q(s, a)$ toward $r + \gamma \max_{a'} Q(s', a')$, but
   the target itself changes every update step. This creates a moving-target problem
   analogous to chasing your own shadow.
   
   **Fix: Target Network** — a second copy of the network whose weights are frozen and
   only updated periodically.

These two ideas are the core contributions of the original DQN paper (Mnih et al., 2015).

## Component 1: Replay Buffer

A circular buffer that stores `(state, action, reward, next_state, done)` tuples.
We sample uniformly at random to break temporal correlations.

In [ ]:
# Purpose: Implement experience replay buffer
# Key Concept: Random sampling from the buffer breaks the time-correlation that breaks SGD

class ReplayBuffer:
    """
    Fixed-size circular buffer for storing and sampling transitions.

    Parameters
    ----------
    capacity : int
        Maximum number of transitions to store. Oldest are evicted first.
    """

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Store one transition."""
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        """
        Sample `batch_size` transitions uniformly at random.

        Returns
        -------
        Tuple of torch.Tensors: (states, actions, rewards, next_states, dones)
        """
        batch = random.sample(self.buffer, batch_size)
        # Unzip list-of-tuples into separate arrays
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            torch.FloatTensor(np.array(states)).to(DEVICE),
            torch.LongTensor(actions).to(DEVICE),
            torch.FloatTensor(rewards).to(DEVICE),
            torch.FloatTensor(np.array(next_states)).to(DEVICE),
            torch.FloatTensor(dones).to(DEVICE),
        )

    def __len__(self):
        return len(self.buffer)


# Quick sanity check
buf = ReplayBuffer(capacity=100)
for _ in range(20):
    buf.push(np.zeros(4), 0, 1.0, np.zeros(4), False)
states, actions, rewards, next_states, dones = buf.sample(batch_size=8)
print(f"Buffer size: {len(buf)}")
print(f"Sampled batch shapes — states: {states.shape}, actions: {actions.shape}, rewards: {rewards.shape}")

## Component 2: Q-Network

The Q-network takes the full state vector as input and outputs one Q-value per action.
This is the **advantage** over tabular Q-learning: one forward pass gives $Q(s, a)$ for
all actions simultaneously, and the network generalises across similar states.

In [ ]:
# Purpose: Define the Q-network architecture
# Key Concept: Output one Q-value per action — argmax gives the greedy action in one forward pass

class QNetwork(nn.Module):
    """
    Three-layer MLP that maps states to Q-values for all actions.

    Parameters
    ----------
    obs_dim : int
        Observation dimension.
    n_actions : int
        Number of discrete actions.
    hidden_dim : int
        Width of hidden layers.
    """

    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),  # no activation — raw Q-values
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Parameters
        ----------
        x : torch.Tensor of shape (batch, obs_dim)

        Returns
        -------
        torch.Tensor of shape (batch, n_actions)
            Raw (unbounded) Q-values.
        """
        return self.net(x)


env = gym.make('CartPole-v1')
obs_dim   = env.observation_space.shape[0]
n_actions = env.action_space.n

# Two identical networks — online is trained every step, target is frozen
online_net = QNetwork(obs_dim, n_actions, hidden_dim=128).to(DEVICE)
target_net = QNetwork(obs_dim, n_actions, hidden_dim=128).to(DEVICE)
target_net.load_state_dict(online_net.state_dict())  # start with same weights
target_net.eval()   # target network is never trained directly

optimizer = optim.Adam(online_net.parameters(), lr=LR)

# Verify output shape
sample_obs, _ = env.reset(seed=0)
q_values = online_net(torch.FloatTensor(sample_obs).to(DEVICE))
print(f"Network parameters: {sum(p.numel() for p in online_net.parameters()):,}")
print(f"Q-values for sample obs: {q_values.detach().cpu().numpy()}")
print(f"Greedy action: {q_values.argmax().item()}")

## Component 3: DQN Agent

The agent wraps action selection (epsilon-greedy using the online network) and the
Bellman update (using the target network for stable targets).

The loss is **Huber loss** (smooth L1), which is less sensitive to large Q-value
errors than MSE — a practical improvement from the original DQN paper.

In [ ]:
# Purpose: DQN agent combining epsilon-greedy selection and Bellman update
# Key Concept: Target network provides stable Bellman targets — online network is updated every step

class DQNAgent:
    """
    Deep Q-Network agent with experience replay and target network.

    Parameters
    ----------
    online_net : QNetwork
        The network trained every step.
    target_net : QNetwork
        Frozen copy used for computing Bellman targets.
    optimizer : torch.optim.Optimizer
    replay_buffer : ReplayBuffer
    batch_size : int
    gamma : float
    epsilon_start : float
    epsilon_end : float
    epsilon_decay : float
    """

    def __init__(self, online_net, target_net, optimizer, replay_buffer,
                 batch_size, gamma, epsilon_start, epsilon_end, epsilon_decay):
        self.online_net    = online_net
        self.target_net    = target_net
        self.optimizer     = optimizer
        self.replay_buffer = replay_buffer
        self.batch_size    = batch_size
        self.gamma         = gamma
        self.epsilon       = epsilon_start
        self.epsilon_end   = epsilon_end
        self.epsilon_decay = epsilon_decay

    def select_action(self, obs: np.ndarray) -> int:
        """
        Epsilon-greedy action selection using the online network.

        Parameters
        ----------
        obs : np.ndarray
            Current environment observation.

        Returns
        -------
        int
        """
        if random.random() < self.epsilon:
            return random.randint(0, n_actions - 1)   # explore
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
            return self.online_net(obs_t).argmax(dim=1).item()  # exploit

    def update(self) -> float:
        """
        Sample a mini-batch and perform one Bellman update.

        Returns
        -------
        float
            Scalar loss value for logging.
        """
        if len(self.replay_buffer) < MIN_BUFFER_SIZE:
            return 0.0   # wait until buffer has enough transitions

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        # --- Online network: Q(s, a) for the taken actions ---
        # gather() picks the Q-value of the action that was actually taken
        current_q = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # --- Target network: max Q(s', a') with no gradient ---
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(dim=1).values
            # If the episode ended (done=1), there is no future return
            target_q = rewards + self.gamma * max_next_q * (1.0 - dones)

        # Huber loss is more robust to large Q-value errors than MSE
        loss = F.smooth_l1_loss(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        return loss.item()

    def sync_target_network(self):
        """Copy online weights to target network."""
        self.target_net.load_state_dict(self.online_net.state_dict())

    def decay_epsilon(self):
        """Multiplicative epsilon decay — called once per episode."""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)


print("DQNAgent class defined.")

## Training Loop — 300 Episodes

300 episodes with CartPole trains in well under a minute on CPU. We track:
- Episode score (steps survived)
- Mean loss per episode
- Epsilon (to see the exploration schedule)
- Buffer fill level (to see when training begins)

In [ ]:
# Purpose: Run the full DQN training loop
# Key Concept: Experience collection and network updates happen at every environment step, not every episode

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

env = gym.make('CartPole-v1')

# Re-initialise all components with fresh seeds
online_net = QNetwork(obs_dim, n_actions, hidden_dim=128).to(DEVICE)
target_net = QNetwork(obs_dim, n_actions, hidden_dim=128).to(DEVICE)
target_net.load_state_dict(online_net.state_dict())
target_net.eval()

optimizer      = optim.Adam(online_net.parameters(), lr=LR)
replay_buffer  = ReplayBuffer(capacity=BUFFER_SIZE)

agent = DQNAgent(
    online_net=online_net,
    target_net=target_net,
    optimizer=optimizer,
    replay_buffer=replay_buffer,
    batch_size=BATCH_SIZE,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_end=EPSILON_END,
    epsilon_decay=EPSILON_DECAY,
)

scores   = []     # episode return
losses   = []     # mean loss per episode
epsilons = []     # epsilon per episode

for episode in range(N_EPISODES):
    obs, _ = env.reset(seed=episode)
    total_reward = 0
    ep_losses    = []
    done = False

    while not done:
        # --- Select and execute action ---
        action = agent.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # --- Store transition in replay buffer ---
        replay_buffer.push(obs, action, reward, next_obs, float(done))

        # --- Update online network ---
        loss = agent.update()
        if loss > 0:
            ep_losses.append(loss)

        obs = next_obs
        total_reward += reward

    # --- End-of-episode bookkeeping ---
    agent.decay_epsilon()

    # Periodically sync the target network
    if (episode + 1) % TARGET_UPDATE == 0:
        agent.sync_target_network()

    scores.append(total_reward)
    losses.append(np.mean(ep_losses) if ep_losses else 0.0)
    epsilons.append(agent.epsilon)

    if (episode + 1) % 50 == 0:
        mean_score = np.mean(scores[-50:])
        print(f"Episode {episode+1:4d} | mean(last 50): {mean_score:6.1f} "
              f"| epsilon: {agent.epsilon:.3f} "
              f"| buffer: {len(replay_buffer):,}/{BUFFER_SIZE:,}")

env.close()
print("\nTraining complete.")

## Training Results

Three panels:
- **Score** — episode return with rolling average
- **Loss** — Huber loss during training (rises initially as Q-values spread out, then
  falls as the network stabilises)
- **Epsilon** — exploration schedule

In [ ]:
# Purpose: Comprehensive visualisation of DQN training dynamics
# Key Concept: All three plots together tell the full story of what happened during training

def rolling_mean(data, window=20):
    return np.convolve(data, np.ones(window) / window, mode='valid')


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ep_range = np.arange(1, N_EPISODES + 1)

# --- Panel 1: Episode scores ---
ax = axes[0]
ax.plot(ep_range, scores, color='#3498db', alpha=0.3, linewidth=0.8)
roll = rolling_mean(scores, window=20)
ax.plot(np.arange(20, N_EPISODES + 1), roll, color='#1a5276', linewidth=2.5,
        label='Rolling mean (20 ep)')
ax.axhline(200, color='#e74c3c', linestyle='--', linewidth=1.2, label='Score 200')
ax.axhline(475, color='#27ae60', linestyle='--', linewidth=1.2, label='Score 475 (solved)')

# Shade the pre-training phase (buffer filling)
fill_ep = np.searchsorted(np.cumsum([1] * N_EPISODES), MIN_BUFFER_SIZE // 10)
ax.axvspan(0, fill_ep, alpha=0.08, color='grey', label='Buffer filling')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Episode Score', fontsize=13)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Panel 2: Loss ---
ax = axes[1]
# Only show episodes where loss was recorded (buffer was full enough)
loss_mask = np.array(losses) > 0
loss_episodes = ep_range[loss_mask]
loss_values   = np.array(losses)[loss_mask]
if len(loss_values) > 0:
    ax.plot(loss_episodes, loss_values, color='#e67e22', alpha=0.4, linewidth=0.8)
    if len(loss_values) >= 20:
        roll_loss = rolling_mean(loss_values, window=20)
        ax.plot(loss_episodes[19:], roll_loss, color='#d35400', linewidth=2.5)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Huber Loss', fontsize=12)
ax.set_title('Training Loss', fontsize=13)
ax.grid(True, alpha=0.3)

# --- Panel 3: Epsilon ---
ax = axes[2]
ax.plot(ep_range, epsilons, color='#8e44ad', linewidth=2)
ax.fill_between(ep_range, epsilons, alpha=0.2, color='#8e44ad')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Epsilon', fontsize=12)
ax.set_title('Exploration Rate', fontsize=13)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.suptitle('DQN Training on CartPole-v1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final 50-episode mean score: {np.mean(scores[-50:]):.1f}")
print(f"Peak episode score:          {max(scores)}")
print(f"Replay buffer transitions:   {len(replay_buffer):,}")

## Clean Evaluation

We run 10 greedy evaluation episodes. The online network is set to `.eval()` mode
and we use `argmax` rather than sampling.

In [ ]:
# Purpose: Measure true post-training performance without exploration noise
# Key Concept: Evaluation performance on held-out seeds is the honest measure of agent quality

online_net.eval()
eval_env = gym.make('CartPole-v1')
eval_scores = []

with torch.no_grad():
    for test_ep in range(10):
        obs, _ = eval_env.reset(seed=3000 + test_ep)
        total = 0
        done  = False
        while not done:
            obs_t  = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
            action = online_net(obs_t).argmax(dim=1).item()
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            done  = terminated or truncated
            total += reward
        eval_scores.append(total)
        print(f"  Eval episode {test_ep+1:2d}: score = {int(total):3d}")

eval_env.close()
print(f"\nMean eval score: {np.mean(eval_scores):.1f} / 500")

## Ablation: Why the Target Network Matters

This diagram shows what would happen without the target network. Without it, the
Bellman targets shift every update step, creating a feedback loop.

With the target network:
```
Online net  <-- Bellman update (every step)
Target net  <-- Copy from online (every TARGET_UPDATE episodes)
```

Without the target network:
```
One net  <-- Bellman update using itself (targets shift every step)
         = chasing your own tail
```

You can verify this by setting `TARGET_UPDATE = 1` (copy every episode) —
training becomes unstable.

In [ ]:
# Purpose: Show Q-value evolution to visualise what the network learned
# Key Concept: Q-values should correlate with actual episode return — a sanity check

# Sample a fixed observation and watch how Q-values evolved conceptually
# (We probe the final trained network at representative states)

probe_env = gym.make('CartPole-v1')
probe_obs, _ = probe_env.reset(seed=999)
probe_env.close()

online_net.eval()
with torch.no_grad():
    obs_t   = torch.FloatTensor(probe_obs).unsqueeze(0).to(DEVICE)
    q_vals  = online_net(obs_t).squeeze().cpu().numpy()

print("Probe observation (cart_pos, cart_vel, pole_angle, pole_angular_vel):")
labels = ['cart position', 'cart velocity', 'pole angle', 'pole angular vel']
for lbl, val in zip(labels, probe_obs):
    print(f"  {lbl:22s}: {val:+.4f}")

print(f"\nQ-value for action 'Left'  (0): {q_vals[0]:.4f}")
print(f"Q-value for action 'Right' (1): {q_vals[1]:.4f}")
print(f"Greedy action: {'Left' if q_vals[0] > q_vals[1] else 'Right'}")

# Visualise
fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(['Left (0)', 'Right (1)'], q_vals,
              color=['#3498db', '#e74c3c'], edgecolor='black', width=0.5)
ax.set_ylabel('Q-value', fontsize=12)
ax.set_title('Trained Q-values for a Sample State', fontsize=13)
for bar, val in zip(bars, q_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Modify and Experiment

Each of these changes teaches something specific about DQN:

1. **Set `TARGET_UPDATE = 1`** — copy every episode. Training should become noisier.
2. **Set `BATCH_SIZE = 8`** — much smaller batches. Does the loss become spikier?
3. **Set `MIN_BUFFER_SIZE = 64`** — start training almost immediately. Does early
   training hurt final performance?
4. **Change loss to `F.mse_loss`** — replace Huber with MSE. Are large early errors
   more disruptive?
5. **Remove gradient clipping** — what happens when Q-values diverge?

## Key Takeaways

1. **Experience replay** breaks temporal correlations that destabilise gradient descent
2. **Target network** provides stable Bellman targets by freezing weights during training
3. **Huber loss** is more robust than MSE for Q-learning because Q-value errors can be
   large early in training
4. **Gradient clipping** prevents any single update from making catastrophic changes
5. **Buffer warmup** (`MIN_BUFFER_SIZE`) ensures the first updates see diverse transitions

## What's Next

DQN on CartPole is the foundation. Extensions that improve it substantially:

- **Double DQN** — use the online network to *select* actions, the target to *evaluate*
  them, reducing Q-value overestimation
- **Dueling DQN** — split the Q-network into value and advantage streams
- **Prioritised Experience Replay** — sample transitions with higher TD-error more
  frequently
- **PPO / A2C** — modern policy gradient methods that outperform DQN on most benchmarks

For a full deep RL treatment, see `modules/module_04_deep_rl/` in this course.